# Gao et al. (2026) on dandiset 001683: seizure metrics from the labels

Paper. Gao H, Frake A, Durand DM, He B (2026). Transcranial focused ultrasound stimulation
targeting white matter inhibits seizures in a rat model of epilepsy. *IEEE Trans Neural Syst Rehabil
Eng* 34, 251-259. [doi:10.1109/tnsre.2025.3644273](https://doi.org/10.1109/tnsre.2025.3644273). The
claims file was written from the NIH author manuscript (PMC12848950) and the supplement.

Dandiset. [DANDI:001683](https://dandiarchive.org/dandiset/001683/0.260109.2104), version
0.260109.2104. The paper cites no version (its Data Availability statement gives the dandiset's
address only). This is the only published version and so the most recent; the draft lists the same
318 assets and size.

This is the first of three notebooks. It tests the claims on seizure count, percent time in
seizure, seizure duration and inter-seizure interval (Figs. 2 and 3), which use only the seizure
labels the dandiset holds. `eeg_blocks.ipynb` tests the band power, asymmetry index and seizure PLI
claims (Figs. 4, 6, S4, S5, S7); `wild_type.ipynb` the evoked potentials and wild-type PLI (Fig. 5).

| Claim | Notebook | Tested here |
|---|---|---|
| fig2c-sham-duration-stable | this | yes |
| fig2d-seizures-under-two-minutes | this | yes |
| fig2e-sham-count-stable | this | yes |
| fig2f-sham-percent-time-post3 | this | yes |
| fig3a-count-reduction | this | yes |
| fig3a-count-sham-vs-tfus | this | yes |
| fig3b-percent-time-reduction | this | yes |
| fig3b-sham-percent-time-no-change | this | yes |
| fig3b-percent-time-sham-vs-tfus | this | yes |
| fig3c-interval-increase | this | yes |
| fig3c-interval-sham-vs-tfus | this | yes |
| fig3d-duration-decline | this | yes |
| Figs. 4, 6, S4, S5, S7 (seven claims) | eeg_blocks.ipynb | no |
| Fig. 5 (three claims) | wild_type.ipynb | no |

In [ ]:
import hashlib
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import tfus

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
assets = tfus.list_assets()
info = tfus.version_info()
print(f"dandiset {tfus.DANDISET_ID} version {tfus.DANDISET_VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e9:.1f} GB; archive summary: {info['metadata']['assetsSummary']['numberOfSubjects']} subjects, "
      f"{info['metadata']['assetsSummary']['numberOfFiles']} files")
big = assets[assets["size"] > 2e9]
metas = tfus.read_all_meta(big.asset_id.tolist())
lab = [m for m in metas if m["kind"] == "Labeled_data"]
print(f"files over 2 GB: {len(big)}; of them labelled EEG blocks: {len(lab)}; read in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
blocks = tfus.assign_blocks(lab)
M = {m["asset_id"]: m for m in lab}
print("Kinds, groups and notes of the files read:")
print(pd.Series([(m["kind"], m["group"], m["notes"]) for m in lab]).value_counts().to_string())
print("\nRats per group:", blocks.groupby("group").subject.nunique().to_dict(), "(paper: 10 tFUS, 6 sham)")
print("\nBlock files per rat, and the label each file carries in general/data_collection, in time order:")
print(blocks.pivot(index="subject", columns="block", values="label").fillna("").to_string())
print("\nSampling rates:", sorted({m["rate"] for m in lab}), "Hz; channels:", sorted({m["n_channels"] for m in lab}),
      "; unit:", sorted({m["unit"] for m in lab}), "; conversion:", sorted({m["conversion"] for m in lab}))
print("Storage: chunks", sorted({str(m["chunks"]) for m in lab}), "compression", sorted({str(m["compression"]) for m in lab}))
print(f"Block length: {blocks.minutes.min():.2f} to {blocks.minutes.max():.2f} min (paper: ~62.5 min)")
gaps = blocks.assign(t=pd.to_datetime(blocks.start, utc=True)).groupby("subject").t.diff().dt.total_seconds() / 60
print(f"Start-to-start time between consecutive blocks of a rat: {gaps.min():.1f} to {gaps.max():.1f} min")

Ten tFUS rats with four blocks each (baseline, CC_1Hz, post1, post2) and six sham rats with five
(baseline, SHAM, post1, post2, post3), as in the paper. Blocks follow each other at 63 to 68 min.
One rat's labels do not match its files: BH635 has two files labelled `BH635_SHAM`, then one
labelled `BH636_post1` (another rat's number) and one `BH635_post2`, and none `post3`. Its five
files are taken here in time order (baseline, condition, post 1, post 2, post 3), as the other
sham rats' five are; the reading that follows the labels instead (the second SHAM file left out,
`BH636_post1` as post 1, `BH635_post2` as post 2, no post 3) is shown as a variant where it
bears on a claim. The session of BH649's last block is named `ses-None` in its file name.

In [ ]:
iv_all = pd.concat([M[a]["intervals"].assign(asset_id=a) for a in blocks.asset_id])
iv_all = iv_all.merge(blocks[["asset_id", "subject", "block", "minutes", "n_samples"]], on="asset_id")
print("Label tags:", iv_all.tag.value_counts().to_dict())
nos = iv_all[iv_all.tag == "no_seizure"]
print("Blocks with a 'no_seizure' row (start, stop):", nos[["subject", "block", "start", "stop"]].to_dict("records"))
real = iv_all[iv_all.tag != "no_seizure"]
print(f"\nLargest stop time in each file, against the block length: stop/length in minutes "
      f"{(real.groupby('asset_id').stop.max() / real.groupby('asset_id').minutes.first()).describe().round(3).to_dict()}")
print(f"Largest stop over all files: {real.stop.max():.2f}; shortest block: {blocks.minutes.min():.2f} min = {blocks.minutes.min() * 60:.0f} s")
print("So the label times are minutes: in seconds every seizure would fall in the first 63 s of a 3750 s block.")
d = (real.stop - real.start) * 60
print(f"Label durations, read as minutes: {d.min():.1f} to {d.max():.1f} s, median {d.median():.1f} s; "
      f"{(d < 10).sum()} under the 10 s minimum of Methods C")
print(f"Rows with stop before start: {(real.stop < real.start).sum()}")
first = real.groupby(["asset_id", "tag"]).start.min()
print(f"Seizures starting at the first sample (start below 2 samples): {(real.start * 60 < 2 / tfus.RATE).sum()}; "
      f"ending at the last sample: {((real.stop * 60) > (real.n_samples / tfus.RATE - 2 / tfus.RATE)).sum()}")
pairs = real.groupby(["asset_id", "tag"]).size().unstack()
print(f"Files whose ipsilateral and contralateral counts differ: {(pairs.ipsilateral != pairs.contralateral).sum()} of {len(pairs)}")
# Resolution of the labels and the distance of each hemisphere's last seizure from the block end
# (verifier, round 1: a seizure cut by the block end may not end at the last sample).
vals = real[["start", "stop"]].to_numpy().ravel()
print(f"\nLabel values (minutes) equal to their rounding to 3, 4 and 6 decimals: "
      f"{', '.join(f'{np.mean(np.isclose(vals, np.round(vals, k), rtol=0, atol=1e-9)):.1%}' for k in (3, 4, 6))}; "
      f"first starts at one sample (1/{tfus.RATE:.0f} s): {int(np.isclose(real.start * 60, 1 / tfus.RATE, atol=1e-6).sum())}")
end_gap = (real.n_samples / tfus.RATE - real.stop * 60).groupby([real.asset_id, real.tag]).min()
print(f"Seconds from each block-hemisphere's last seizure end to the block end: smallest {np.sort(end_gap.to_numpy())[:8].round(2).tolist()}; "
      f"within 2 s: {int((end_gap < 2).sum())} of {len(end_gap)}; within 10 s: {int((end_gap < 10).sum())}")

The label table gives start and stop times in minutes, not in the seconds NWB prescribes. The
notebook `eeg_blocks.ipynb` checks this against the EEG: the 30 to 80 Hz power of the labelling
channels is compared inside and outside the labels, with the labels read as minutes and shifted
in time. Every metric here uses the labels in seconds after multiplying by 60.

In [ ]:
# Recordings deposited twice: identical label tables, sample counts, or identical samples.
sig = {}
for a in blocks.asset_id:
    m = M[a]
    x = tfus.read_rows(m, 2_000_000, 2_002_000)
    sig[a] = (hashlib.md5(x.tobytes()).hexdigest(), m["n_samples"],
              hashlib.md5(m["intervals"].round(6).to_csv().encode()).hexdigest())
s = pd.DataFrame(sig, index=["samples_md5", "n_samples", "labels_md5"]).T
print("Files sharing 2000 samples at row 2,000,000:", int(s.samples_md5.duplicated(keep=False).sum()))
dl = s[s.labels_md5.duplicated(keep=False)].join(blocks.set_index("asset_id")[["subject", "block"]])
print("Files sharing a label table:", len(dl), dl[["subject", "block"]].to_dict("records"),
      "; their tables:", [M[a]["intervals"].to_dict("records") for a in dl.index])
dup_n = s[s.n_samples.duplicated(keep=False)].join(blocks.set_index("asset_id")[["subject", "block"]])
print("Files sharing a sample count (same length is common at this rate; listed for inspection):")
print(dup_n.sort_values("n_samples")[["n_samples", "subject", "block"]].to_string())
print("read_rows agrees with h5py on a slice:", tfus.check_read(lab[0]))

No two block files share samples, so no recording is deposited twice. The two files that share a
label table are the two blocks without seizures, whose tables hold the same single `no_seizure`
row. Files that share a sample count differ in their samples.

## Seizure metrics

Per rat, hemisphere and block (M2 of the claims file, Methods C): Seizure Count; Percent Time in
Seizure, summed durations over the block length; Seizure Duration, the mean duration of seizures
that neither start at the first sample nor end at the last; and Inter-Seizure Interval, the mean
interval between neighbouring seizures, computed both as the gap from one seizure's end to the next
one's start (`isi_gap`) and from onset to onset (`isi_onset`). The paper's words, "time between
neighboring seizures", favour the gap. For the interval the two tFUS rats with a block without
seizures are left out (Methods C, "two rats were excluded because they had conditions without
seizure occurrence").

In [ ]:
def metrics_table(blocks, edge=1.5 / tfus.RATE):
    rows = []
    for _, r in blocks.iterrows():
        for h in ("ipsilateral", "contralateral"):
            rows.append({"subject": r.subject, "group": r.group, "block": r.block, "hemi": h,
                         **tfus.block_metrics(M[r.asset_id], h, edge)})
    return pd.DataFrame(rows)

met = metrics_table(blocks)
no_seiz = sorted(met[(met["count"] == 0)].subject.unique())
print("Rat-blocks without seizures:", met[met["count"] == 0][["subject", "group", "block", "hemi"]].to_dict("records"))
print("Rats left out of the inter-seizure interval:", no_seiz, "(Methods C: two rats; Fig. 3C: N = 8 tFUS)")
met.loc[met.subject.isin(no_seiz), ["isi_gap", "isi_onset"]] = np.nan
print("Truncated seizures left out of the duration:", int(met.n_truncated.sum()))
print("\nGroup means per hemisphere and block:")
print(met.groupby(["group", "hemi", "block"])[["count", "percent_time", "duration", "isi_gap", "isi_onset"]].mean().round(2).to_string())
print("\nRanges over rat-blocks:", {c: (round(met[c].min(), 2), round(met[c].max(), 2)) for c in ["count", "percent_time", "duration", "isi_gap", "isi_onset"]})

# The label-following reading of BH635's blocks.
alt = blocks.copy()
b635 = alt[alt.subject == "BH635"].sort_values("start").index
alt = alt.drop(index=b635[2])
alt.loc[b635[3], "block"], alt.loc[b635[4], "block"] = "post1", "post2"
met_alt = metrics_table(alt)
met_alt.loc[met_alt.subject.isin(no_seiz), ["isi_gap", "isi_onset"]] = np.nan

# Variant for the durations: seizures within 2 s of either end of the block counted as cut off.
met_edge = metrics_table(blocks, edge=2.0)
print("\nSeizures left out of the duration with a 2 s edge:", int(met_edge.n_truncated.sum()))

In [ ]:
# Statistics: two-way ANOVA and Bonferroni post hoc tests, under the readings of M3.
BL = ["baseline", "condition", "post1", "post2", "post3"]
NAMES = {"baseline": "baseline", "condition": "condition", "post1": "post 1", "post2": "post 2", "post3": "post 3"}

def sham_tests(df, value):
    # Fig. 2: hemisphere by block, sham rats, five blocks. Post hoc: each pair of blocks within each
    # hemisphere (family 'blocks', 20 comparisons) and the hemispheres at each block (family 'hemi', 5).
    comps = []
    for h in ("ipsilateral", "contralateral"):
        for i, b1 in enumerate(BL):
            for b2 in BL[i + 1:]:
                comps.append((f"{h[:6]} {b1}-{b2}", {"hemi": h, "block": b1}, {"hemi": h, "block": b2}, "blocks"))
    for b in BL:
        comps.append((f"ipsi-contra {b}", {"hemi": "ipsilateral", "block": b}, {"hemi": "contralateral", "block": b}, "hemi"))
    out = {}
    for rm in (False, True):
        table, post = tfus.twoway(df, value, "hemi", "block", "subject", comps, rm=rm)
        out["repeated" if rm else "ordinary"] = (table, post)
    return out

def group_tests(df, value, hemi):
    # Fig. 3 and 4: group by block per hemisphere, blocks baseline to post 2. Families (M3):
    # 'within' each block against baseline in each group (6), 'between' sham against tFUS at each block (4).
    d = df[(df.hemi == hemi) & (df.block != "post3")]
    comps = [(f"{g} baseline-{b}", {"group": g, "block": "baseline"}, {"group": g, "block": b}, "within")
             for g in ("sham", "tFUS") for b in BL[1:4]]
    comps += [(f"sham-tFUS {b}", {"group": "sham", "block": b}, {"group": "tFUS", "block": b}, "between") for b in BL[:4]]
    out = {}
    for rm in (False, True):
        table, post = tfus.twoway(d, value, "group", "block", "subject", comps, rm=rm)
        out["repeated" if rm else "ordinary"] = (table, post)
    return out

def show(out, keep=None):
    for k, (table, post) in out.items():
        print(f"  [{k}] main effects:", {i: f"F={r.F:.2f}, p={r.p:.3g}" for i, r in table.iterrows()})
        p = post if keep is None else post[post.comparison.isin(keep)]
        print(p[["comparison", "family", "diff", "stat", "p_raw", "p_family", "p_all"]].round(4).to_string(index=False))

def pct_change(df, value, group="tFUS"):
    # M4: (a) the group mean of the block over the group mean at baseline, minus 1;
    # (b) the mean over rats of each rat's change from its own baseline.
    d = df[df.group == group]
    g = d.groupby(["hemi", "block"])[value].mean().unstack()
    a = (g.div(g["baseline"], axis=0) - 1) * 100
    w = d.pivot_table(index=["hemi", "subject"], columns="block", values=value)
    b = ((w.div(w["baseline"], axis=0) - 1) * 100).groupby("hemi").mean()
    cols = [c for c in BL[1:4]]
    return a[cols], b[cols]

## fig2c-sham-duration-stable

> In the sham group, seizure durations from the ipsilateral and contralateral hemispheres do not
> differ significantly across time blocks or between hemispheres.

A claim of absence: two-way ANOVA (hemisphere by block) with Bonferroni post hoc tests at
P < 0.05, with one value per seizure (reading a, as the Fig. 2C legend says each point is one
seizure) or one mean per rat (reading b), each under the ordinary and repeated-measures readings.

In [ ]:
sham = blocks[blocks.group == "sham"]
seiz = []
for _, r in sham.iterrows():
    for h in ("ipsilateral", "contralateral"):
        s = tfus.seizures(M[r.asset_id], h)
        s = s[~s.truncated]
        seiz.append(s.assign(subject=r.subject, block=r.block, hemi=h))
seiz = pd.concat(seiz, ignore_index=True)
print(f"Sham seizures: {len(seiz)} (ipsilateral {sum(seiz.hemi == 'ipsilateral')}, contralateral {sum(seiz.hemi == 'contralateral')}); "
      f"duration {seiz.duration.min():.1f} to {seiz.duration.max():.1f} s")
print(seiz.groupby(["hemi", "block"]).duration.agg(["size", "mean", "median"]).round(1).unstack(0).to_string())
res2c = {}
for reading, df in (("a, per seizure", seiz), ("b, per-rat means", met[met.group == "sham"])):
    print(f"\nReading {reading}:")
    out = sham_tests(df, "duration")
    res2c[reading] = out
    for k, (table, post) in out.items():
        sig = post[(post.p_family < 0.05) | (post.p_all < 0.05)]
        print(f"  [{k}] main effects:", {i: f"F={r.F:.2f}, p={r.p:.3g}" for i, r in table.iterrows()})
        print(f"  [{k}] post hoc significant (own family or all): {len(sig)} of {len(post)}")
        print(sig[["comparison", "diff", "p_raw", "p_family", "p_all"]].round(4).to_string(index=False) if len(sig) else "  none")
print("\nVariant, seizures within 2 s of a block end left out (per-rat means):")
for k, (table, post) in sham_tests(met_edge[met_edge.group == "sham"], "duration").items():
    print(f"  [{k}] main effects:", {i: f"F={r.F:.2f}, p={r.p:.3g}" for i, r in table.iterrows()},
          "; post hoc significant (own family):", int((post.p_family < 0.05).sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for k, h in enumerate(("ipsilateral", "contralateral")):
    d = seiz[seiz.hemi == h]
    x = d.block.map({b: i for i, b in enumerate(BL)}) + (k - 0.5) * 0.3
    ax.scatter(x, d.duration, s=4, alpha=0.4, label=h)
ax.set_xticks(range(5), BL); ax.set_ylabel("seizure duration (s)"); ax.legend()
ax.set_title(f"Sham seizures, all points shown ({seiz.duration.min():.0f} to {seiz.duration.max():.0f} s)")
plt.tight_layout(); plt.show()

**Result: not reproduced.** The effect of block on sham seizure duration is significant under
every reading: per seizure, F = 30.45, p = 5.5e-24 (ordinary) and F = 37.48, p = 2.7e-29
(repeated), with 12 and 13 of 25 post hoc comparisons significant; per rat, p = 0.00145 (ordinary,
no post hoc comparison significant) and p = 0.00037 (repeated, contralateral baseline against
post 3 significant, p = 0.026). The hemisphere effect is not significant under any reading
(p = 0.16 to 0.63). Mean durations fall from 49.6 to 53.0 s at baseline to 34.8 to 35.7 s at
post 3. Counting the seizures that end within 2 s of a block end as cut off changes nothing (per
rat, block p = 0.0017 ordinary and 0.00043 repeated).

## fig2d-seizures-under-two-minutes

> Pooled over blocks, most seizures in the sham group last less than 2 minutes, and the two
> hemispheres have similar distributions.

Seizures of the six sham rats, all five blocks pooled, per hemisphere. Reproduced when more than
half last under 120 s in each hemisphere. Medians and a two-sample Kolmogorov-Smirnov test are
description only.

In [ ]:
allseiz = []
for _, r in sham.iterrows():
    for h in ("ipsilateral", "contralateral"):
        allseiz.append(tfus.seizures(M[r.asset_id], h).assign(subject=r.subject, block=r.block, hemi=h))
allseiz = pd.concat(allseiz, ignore_index=True)
for h in ("ipsilateral", "contralateral"):
    d = allseiz[allseiz.hemi == h].duration
    print(f"{h}: {len(d)} seizures, {100 * (d < 120).mean():.1f}% under 120 s, median {d.median():.1f} s, range {d.min():.1f} to {d.max():.1f} s")
    print("   per rat, % under 120 s:", (allseiz[allseiz.hemi == h].groupby("subject").duration.apply(lambda x: round(100 * (x < 120).mean(), 1))).to_dict())
ks = stats.ks_2samp(allseiz[allseiz.hemi == "ipsilateral"].duration, allseiz[allseiz.hemi == "contralateral"].duration)
print(f"Kolmogorov-Smirnov ipsilateral against contralateral: D = {ks.statistic:.3f}, p = {ks.pvalue:.3g}")
print(f"Excluding truncated seizures changes the counts by {int(allseiz.truncated.sum())}")
fig, ax = plt.subplots(figsize=(6, 3))
bins = np.arange(0, 241, 6)
for h in ("ipsilateral", "contralateral"):
    ax.hist(allseiz[allseiz.hemi == h].duration, bins=bins, alpha=0.5, label=h)
ax.axvline(120, color="k", lw=0.8); ax.set_xlabel("seizure duration (s)"); ax.legend()
print(f"Seizures beyond the 240 s axis: {(allseiz.duration > 240).sum()}")
plt.tight_layout(); plt.show()

**Result: reproduced.** 99.4% of the 527 ipsilateral and 99.2% of the 521 contralateral sham
seizures last under 120 s, with medians of 41.5 and 43.5 s; every rat is at 97% or more. The KS test
(D = 0.082, p = 0.056) is description and does not decide.

## fig2e-sham-count-stable

> In the sham group, seizure count does not differ significantly between time blocks or
> hemispheres, with a slight decrease in the last two hours.

A claim of absence: Seizure Count per rat, hemisphere and block, five blocks, two-way ANOVA
(hemisphere by block) with Bonferroni post hoc tests at P < 0.05.

In [ ]:
s = met[met.group == "sham"]
print("Mean count (rats) per hemisphere and block:")
print(s.groupby(["hemi", "block"])["count"].agg(["mean", "sem"]).round(2).unstack(0).to_string())
print("\nPer rat (ipsilateral):")
print(s[s.hemi == "ipsilateral"].pivot(index="subject", columns="block", values="count").to_string())
out = sham_tests(s, "count")
for k, (table, post) in out.items():
    sig = post[(post.p_family < 0.05) | (post.p_all < 0.05)]
    print(f"[{k}] main effects:", {i: f"F={r.F:.2f}, p={r.p:.3g}" for i, r in table.iterrows()},
          f"; significant post hoc: {len(sig)} of {len(post)}; smallest p (own family) {post.p_family.min():.3g}")
early = s[s.block.isin(BL[:3])]["count"].mean(); late = s[s.block.isin(BL[3:])]["count"].mean()
print(f"\nMean count, baseline to post 1: {early:.2f}; post 2 and post 3: {late:.2f}")
s_alt = met_alt[met_alt.group == "sham"]
out_alt = sham_tests(s_alt, "count")
print("Variant, BH635 blocks by their labels:", {k: f"block p={t.iloc[1].p:.3g}, smallest post hoc p {p.p_family.min():.3g}" for k, (t, p) in out_alt.items()})

**Result: reproduced.** No main effect is significant (block p = 0.97 ordinary and 0.82 repeated;
hemisphere p = 0.93 and 0.88) and no post hoc comparison is (smallest Bonferroni p = 1). The mean
count is 18.11 over baseline to post 1 and 16.50 over post 2 and post 3, the slight decrease the
paper describes. With BH635's blocks taken by their labels the block effect stays non-significant
(p = 0.66 and 0.61).

## fig2f-sham-percent-time-post3

> In the sham group, percent time in seizure in the contralateral hemisphere differs significantly
> between the sham condition block and post 3, suggesting the model is no longer stable in the
> fifth hour.

Percent Time in Seizure, sham rats, five blocks, hemisphere by block. "sham" read as (a) the
condition block or (b) the baseline block. Reproduced when, under one reading, the contralateral
comparison with post 3 is significant at P < 0.05.

In [ ]:
print("Mean percent time (rats) per hemisphere and block:")
print(s.groupby(["hemi", "block"])["percent_time"].agg(["mean", "sem"]).round(2).unstack(0).to_string())
keep = ["contra condition-post3", "contra baseline-post3", "ipsila condition-post3", "ipsila baseline-post3"]
for name, df in (("blocks in time order", s), ("variant, BH635 by labels (no post 3 for BH635)", s_alt)):
    print(f"\n{name}:")
    show(sham_tests(df, "percent_time"), keep)

**Result: reproduced, resting on assumptions.** Under the repeated-measures reading the
contralateral comparison of baseline (reading b of "sham") with post 3 is significant in its
family (Bonferroni p = 0.040), a decrease from 25.6% to 15.5%; over all 25 comparisons it is not
(p = 0.051), and condition against post 3 is not (p = 0.14). Under the ordinary reading no
comparison is significant (p = 1). The result rests on the repeated-measures reading, the family
reading and on taking BH635's files in time order: with BH635 by its labels the comparison is not
significant (p = 0.13 repeated, 0.94 ordinary).

## Figure 3: tFUS against sham

Each metric is tested per hemisphere with a group by block two-way ANOVA (baseline, condition,
post 1, post 2; post 3 left out, Results A), with Bonferroni post hoc tests: each block against
baseline within each group (family `within`, six comparisons) and sham against tFUS at each block
(family `between`, four); `p_all` corrects over all ten. Percent changes are given under both
readings of M4.

In [ ]:
def fig3(value, df=met):
    a, b = pct_change(df, value)
    print(f"{value}: tFUS change from baseline, %, reading (a) ratio of group means:")
    print(a.round(1).to_string())
    print(f"reading (b) mean of each rat's change:")
    print(b.round(1).to_string())
    print("Group means and SEM:")
    print(df[df.block != "post3"].groupby(["hemi", "group", "block"])[value].agg(["mean", "sem", "count"]).round(2).unstack(1).to_string())
    res = {}
    for h in ("ipsilateral", "contralateral"):
        print(f"\n{h}:")
        res[h] = group_tests(df, value, h)
        show(res[h])
    return a, b, res

## fig3a-count-reduction

> In the tFUS group, seizure count fell from baseline by 9%, 36% and 42% in the ipsilateral S1
> and by 10%, 41% and 42% in the contralateral S1 during stimulation, post 1 and post 2, with
> significant reductions from baseline to post 1 and to post 2 in both hemispheres.

In [ ]:
a, b, res3a = fig3("count")
exp = {("ipsilateral", "condition"): 9, ("ipsilateral", "post1"): 36, ("ipsilateral", "post2"): 42,
       ("contralateral", "condition"): 10, ("contralateral", "post1"): 41, ("contralateral", "post2"): 42}
for name, t in (("a", a), ("b", b)):
    rows = [{"hemi": h, "block": k, "expected": v, "measured decrease": round(-t.loc[h, k], 1),
             "within 10%": abs(-t.loc[h, k] - v) <= 0.1 * v} for (h, k), v in exp.items()]
    print(f"\nReading ({name}) of M4:"); print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under reading (b) of M4, each rat's change from its own baseline, the six
decreases are 8.7%, 36.4% and 41.9% (ipsilateral) and 10.5%, 41.4% and 42.4% (contralateral), all
within tolerance of 9, 36, 42, 10, 41 and 42; reading (a) gives 24 to 50%, outside. Under the
repeated-measures reading baseline against post 1 and against post 2 are significant in both
hemispheres (Bonferroni p = 0.016 and 0.0087 ipsilateral, 0.011 and 0.0059 contralateral, in the
`within` family; 0.027, 0.014, 0.019 and 0.0098 over all ten). Under the ordinary reading none of
the four is significant (p = 0.057 to 0.115).

## fig3a-count-sham-vs-tfus

> Seizure count in the sham group stays steady, and the sham and tFUS groups differ significantly
> during post 1 and post 2.

From the same tests: sham against tFUS at post 1 and post 2 (in both hemispheres, reading a, or
either, reading b), and no sham block different from sham baseline.

In [ ]:
for h in ("ipsilateral", "contralateral"):
    for k, (table, post) in res3a[h].items():
        p = post[post.comparison.isin(["sham-tFUS post1", "sham-tFUS post2", "sham baseline-condition", "sham baseline-post1", "sham baseline-post2"])]
        print(h, k, {r.comparison: (round(r.diff, 2), round(r.p_family, 4), round(r.p_all, 4)) for r in p.itertuples()})

**Result: partially reproduced.** Sham against tFUS is significant at post 1 in both hemispheres
under both ANOVA readings with the `between` family (ordinary p = 0.035 ipsilateral and 0.025
contralateral; repeated 0.040 and 0.029), and at post 2 in neither (ordinary 0.057 and 0.069;
repeated 0.064 and 0.076). No sham block differs from sham baseline (p = 1 under both readings), so
the sham count is steady. Only one of post 1 and post 2 is significant, the criterion's partial
outcome. With BH635's blocks by their labels the pattern turns over (post 2 significant, 0.038 to
0.048; post 1 not, 0.055 to 0.062), still one of the two. The first version of this attempt
reported ipsilateral post 2 as significant (p = 0.046) under the repeated reading; that p came from
a normal reference in place of the t distribution (see the summary).

## fig3b-percent-time-reduction

> In the tFUS group, percent time in seizure decreased significantly from baseline by 33%
> (ipsilateral) and 35% (contralateral) during stimulation and by 55% in post 1 and post 2 in both
> hemispheres.

In [ ]:
a, b, res3b = fig3("percent_time")
exp = {("ipsilateral", "condition"): 33, ("contralateral", "condition"): 35,
       ("ipsilateral", "post1"): 55, ("contralateral", "post1"): 55,
       ("ipsilateral", "post2"): 55, ("contralateral", "post2"): 55}
for name, t in (("a", a), ("b", b)):
    rows = [{"hemi": h, "block": k, "expected": v, "measured decrease": round(-t.loc[h, k], 1),
             "within 10%": abs(-t.loc[h, k] - v) <= 0.1 * v} for (h, k), v in exp.items()]
    print(f"\nReading ({name}) of M4:"); print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** The decreases under reading (b) of M4 are 32.5% and 34.8% (stimulation),
55.5% and 55.3% (post 1), 54.9% and 55.2% (post 2), ipsilateral and contralateral, all within
tolerance; reading (a) gives 35.8% to 57.3%, also within. Under the repeated-measures reading all
six comparisons with baseline are significant (Bonferroni p from 0.0025 down to below 0.0001), so
reading (a) of "significantly" holds; under the ordinary reading post 1 and post 2 are (p = 0.008
to 0.012) and stimulation is not (p = 0.22 and 0.17).

## fig3b-sham-percent-time-no-change

> In the sham group, percent time in seizure decreased slightly but did not differ significantly
> from baseline in any block.

A claim of absence: no sham block (condition, post 1, post 2) differs from sham baseline at
P < 0.05 in either hemisphere.

In [ ]:
for h in ("ipsilateral", "contralateral"):
    for k, (table, post) in res3b[h].items():
        p = post[post.comparison.str.startswith("sham baseline")]
        print(h, k, {r.comparison: (round(r.diff, 2), round(r.p_family, 4), round(r.p_all, 4)) for r in p.itertuples()})

**Result: not reproduced, resting on an assumption.** The two ANOVA readings disagree, and so do the
two family readings. Under the ordinary reading no sham block differs from baseline (smallest
Bonferroni p = 0.79). Under the repeated-measures reading, adopted here as the paper's test with
the `within` family (see the summary), contralateral post 2 differs from baseline (p = 0.034;
ipsilateral post 2 p = 0.062); over all ten comparisons it does not (p = 0.057). The sham percent
time falls from 24.0% to 16.4% (ipsilateral) and from 25.6% to 17.6% (contralateral). With BH635's
blocks by their labels, contralateral post 2 is significant under the repeated reading (p = 0.024)
and not under the ordinary one (p = 0.53).

In [ ]:
# Variant: BH635's blocks by their labels (seizure_metrics data check). Sham comparisons and sham against
# tFUS, both ANOVA readings.
for k in ("ordinary", "repeated"):
    for value in ("count", "percent_time", "isi_gap", "duration"):
        for h in ("ipsilateral", "contralateral"):
            table, post = group_tests(met_alt, value, h)[k]
            p = post[post.comparison.str.startswith("sham baseline") | (post.family == "between")]
            print(k, value, h, {r.comparison: round(r.p_family, 4) for r in p.itertuples()})

## fig3b-percent-time-sham-vs-tfus

> Percent time in seizure differs significantly between sham and tFUS during post 1 in the
> contralateral S1, and during stimulation and post 1 in the ipsilateral S1.

In [ ]:
want = [("contralateral", "sham-tFUS post1"), ("ipsilateral", "sham-tFUS condition"), ("ipsilateral", "sham-tFUS post1")]
for k in ("ordinary", "repeated"):
    rows = []
    for h, c in want:
        r = res3b[h][k][1].set_index("comparison").loc[c]
        rows.append({"hemi": h, "comparison": c, "sham minus tFUS": round(r["diff"], 2), "p_family": round(r.p_family, 4), "p_all": round(r.p_all, 4)})
    print(k); print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** tFUS is lower than sham in all three named comparisons (by 11.4,
9.8 and 10.0 percentage points), but none is significant under either reading: contralateral
post 1 p = 0.075 (ordinary) and 0.097 (repeated), ipsilateral stimulation 0.14 and 0.16,
ipsilateral post 1 0.13 and 0.15, in the `between` family.

## fig3c-interval-increase

> Relative to baseline, the inter-seizure interval of the tFUS group increased by 27%, 49% and
> 28% (ipsilateral) and 30%, 52% and 31% (contralateral) during stimulation, post 1 and post 2;
> only post 1 in the contralateral S1 differed significantly from baseline.

The 8 tFUS rats left after the Methods' exclusion. Both readings of the interval are run; the gap
reading is the one the paper's words favour.

In [ ]:
print("Rats in the interval analysis per group:", met.dropna(subset=["isi_gap"]).groupby("group").subject.nunique().to_dict())
exp = {("ipsilateral", "condition"): 27, ("ipsilateral", "post1"): 49, ("ipsilateral", "post2"): 28,
       ("contralateral", "condition"): 30, ("contralateral", "post1"): 52, ("contralateral", "post2"): 31}
res3c = {}
for value in ("isi_gap", "isi_onset"):
    print(f"\n==== {value}")
    a, b, res3c[value] = fig3(value)
    for name, t in (("a", a), ("b", b)):
        rows = [{"hemi": h, "block": k, "expected": v, "measured increase": round(t.loc[h, k], 1),
                 "within 10%": abs(t.loc[h, k] - v) <= 0.1 * v} for (h, k), v in exp.items()]
        print(f"Reading ({name}) of M4:"); print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced, resting on an assumption.** With the interval read as the gap between one
seizure's end and the next one's start, and reading (b) of M4, the increases are 27.4%, 49.1% and
27.8% (ipsilateral) and 30.1%, 51.9% and 30.9% (contralateral), all within tolerance. Under the
repeated-measures reading contralateral post 1 is the only significant comparison with baseline
under both family readings (p = 0.0086 in the `within` family, 0.014 over all ten); ipsilateral
post 1 is not (0.056 and 0.093), and the other four have p = 0.70 or more. The ordinary reading
finds none (contralateral post 1 p = 0.18). The onset-to-onset reading gives increases of 12.6% to
30.6%, outside tolerance, so the outcome rests on the gap reading, which the paper's "time between
neighboring seizures" favours.

## fig3c-interval-sham-vs-tfus

> The inter-seizure interval of the sham group stays steady, and sham and tFUS differ
> significantly during post 1 in both hemispheres.

In [ ]:
for value in ("isi_gap", "isi_onset"):
    for h in ("ipsilateral", "contralateral"):
        for k, (table, post) in res3c[value][h].items():
            p = post[post.comparison.isin(["sham-tFUS post1", "sham baseline-condition", "sham baseline-post1", "sham baseline-post2"])]
            print(value, h, k, {r.comparison: (round(r.diff, 1), round(r.p_family, 4), round(r.p_all, 4)) for r in p.itertuples()})

**Result: partially reproduced.** Sham against tFUS at post 1 is significant in the contralateral
S1 with tFUS higher (by 140 s; ordinary p = 0.027, repeated 0.037, `between` family) and not in
the ipsilateral S1 (by 122 s; ordinary 0.051, repeated 0.061); the onset-to-onset interval gives
the same (contralateral 0.029 and 0.038, ipsilateral 0.058 and 0.066). No sham block differs from
sham baseline (p = 1), so the sham interval is steady. The claim holds in one hemisphere, the
criterion's partial outcome. The first version of this attempt reported the ipsilateral comparison
as significant (p = 0.039) under the repeated reading; that p came from a normal reference in place
of the t distribution.

## fig3d-duration-decline

> Seizure duration declines across time in both groups, and the only significant reduction is
> from baseline to post 2 in both hemispheres of the tFUS group.

In [ ]:
a, b, res3d = fig3("duration")
for h in ("ipsilateral", "contralateral"):
    for k, (table, post) in res3d[h].items():
        p = post[post.family == "within"]
        print(h, k, "within-group against baseline:", {r.comparison: (round(r.diff, 1), round(r.p_family, 4)) for r in p.itertuples()})
print("\nGroup means at baseline and post 2:")
print(met[met.block.isin(["baseline", "post2"])].groupby(["group", "hemi", "block"]).duration.mean().round(1).unstack().to_string())
print("\nVariant, seizures within 2 s of a block end left out:")
for h in ("ipsilateral", "contralateral"):
    for k, (table, post) in group_tests(met_edge, "duration", h).items():
        p = post[post.family == "within"]
        print(h, k, {r.comparison: round(r.p_family, 4) for r in p.itertuples()})

**Result: not reproduced.** The group means decline from baseline to post 2 in both groups
(tFUS 59.8 to 48.7 s ipsilateral and 61.0 to 49.9 s contralateral; sham 53.0 to 37.8 s and 56.8 to
40.6 s), but tFUS baseline against post 2 is not significant in either hemisphere (Bonferroni
p = 0.42 and 0.33 repeated, 0.62 and 0.59 ordinary), nor is any other within-group comparison
against baseline (smallest p = 0.16). Counting the seizures that end within 2 s of a block end as
cut off gives the same (tFUS post 2 p = 0.42 and 0.33 repeated).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Notebook | Outcome |
|---|---|---|
| fig2c-sham-duration-stable | seizure_metrics | not reproduced |
| fig2d-seizures-under-two-minutes | seizure_metrics | reproduced |
| fig2e-sham-count-stable | seizure_metrics | reproduced |
| fig2f-sham-percent-time-post3 | seizure_metrics | reproduced (assumed) |
| fig3a-count-reduction | seizure_metrics | reproduced |
| fig3a-count-sham-vs-tfus | seizure_metrics | partially reproduced |
| fig3b-percent-time-reduction | seizure_metrics | reproduced |
| fig3b-sham-percent-time-no-change | seizure_metrics | not reproduced (assumed) |
| fig3b-percent-time-sham-vs-tfus | seizure_metrics | not reproduced |
| fig3c-interval-increase | seizure_metrics | reproduced (assumed) |
| fig3c-interval-sham-vs-tfus | seizure_metrics | partially reproduced |
| fig3d-duration-decline | seizure_metrics | not reproduced |
| fig4-no-theta-alpha-gamma-power-change | eeg_blocks | not reproduced (assumed) |
| fig4c-beta-decrease | eeg_blocks | partially reproduced |
| figs4-sham-power-no-change | eeg_blocks | not reproduced (assumed) |
| fig4ef-ai-increase | eeg_blocks | partially reproduced |
| figs5-seizures-lower-ai | eeg_blocks | reproduced (assumed) |
| fig6-epileptic-pli-s1-decrease | eeg_blocks | not reproduced |
| figs7-seizure-pli-s1-vs-non-s1 | eeg_blocks | partially reproduced (assumed) |
| fig5b-tfus-evoked-potentials | wild_type | blocked: methods underspecified |
| fig5cdj-wild-type-gamma-pli-decrease | wild_type | blocked: data missing |
| fig5ghi-low-band-pli-s1-only | wild_type | not reproduced |

How the analysis differs from the paper's. The seizure labels are the dandiset's own, so the
metrics rest on the paper's labelling. The label times are in minutes and are converted. The
percent changes of Fig. 3 match the paper to within one point under reading (b) of M4 (each rat's
change from its own baseline), and the inter-seizure interval matches only when read as the gap
from one seizure's end to the next one's start, so both readings are the paper's. The paper's
two-way ANOVA is not specified further. Its significant within-group changes of Fig. 3 (count and
percent time against baseline, the interval at contralateral post 1) appear only under the
repeated-measures reading, implemented here as a mixed model with a random intercept per rat, so
that reading is adopted as an assumption where the readings disagree on a claim of absence. Its
post hoc tests are t tests and its main effects F tests, on Satterthwaite degrees of freedom (on
balanced data, the df of the classical repeated-measures ANOVA, as Prism uses); the first version
of this attempt used normal and chi-squared references, which made two sham against tFUS
comparisons look significant. The Bonferroni family is not specified either; both readings are
shown. Each set of comparisons as its own family (reading i) is adopted where they disagree, as
Prism corrects each set of multiple comparisons it is asked for on its own; this decides
fig2f-sham-percent-time-post3 and fig3b-sham-percent-time-no-change. The durations leave out
seizures that start or end at a block edge; counting the six that end 1 to 1.4 s before a block
end as cut off as well changes no result. BH635's five sham files are taken in time order though their
labels say otherwise.

What was found about the dataset. It holds the paper's 16 epileptic rats (10 tFUS, 6 sham)
with the seizure labels, and the percent changes computed from them match the paper's. The label
table gives times in minutes, not the seconds NWB prescribes (the EEG check in `eeg_blocks.ipynb`
confirms it). BH635's block files are mislabelled (two `SHAM` files, one named for rat BH636, no
`post3`), and BH649's post 3 file has the session id `None`. Every file gives the strain as
"HSD:WI rat" (Wistar) and the age as P24W, while the paper used Sprague-Dawley rats of 3 to 6
months. The labelling channels are not named in the files.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")